# Customer Dimension Processing (Sports Bar)

This notebook processes customer data from the acquired company (Sports Bar) and integrates it into the unified FMCG Lakehouse.

The pipeline ingests raw customer data, applies data quality and standardization rules, and produces
Bronze, Silver, and Gold customer dimension tables aligned with the parent company’s data model.


## Import Required Libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

## Load Shared Utilities and Runtime Parameters

Reusable configuration values are loaded from the utilities notebook.
Widgets are used to parameterize the catalog and data source for reusability.


In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-final/{data_source}/*.csv'
print(base_path)

## Bronze Layer – Raw Customer Data Ingestion

Raw customer data is ingested from the S3 landing zone with minimal transformation.
Metadata columns are added to support traceability.


In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
# print check data type
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Layer – Data Cleansing and Standardization

The Silver layer applies data quality rules and business transformations
to produce a clean and standardized customer dataset.


In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

### Applied Transformations


#### Deduplicate Customers

Duplicate customer records are removed based on customer identifiers.


In [0]:
df_silver = df_bronze.dropDuplicates(['customer_id'])

#### Trim Whitespace from Customer Names


In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

#### Correct City Name Inconsistencies

City names are standardized using predefined mappings and allowed values.


In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

# typos → correct names
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

#### Normalize Customer Name Casing


In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

#### Handle Missing Cities Using Business-Approved Fixes

For specific customers, missing city values are corrected using
business-validated mappings.


In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

#### Standardize Customer Identifier Data Type


In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))

## Align Customer Attributes with Parent Company Data Model

Customer attributes are standardized to ensure compatibility with the parent
company’s analytical schema.


In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

## Persist Silver Customer Dimension


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

## Gold Layer – Business-Ready Customer Dimension


The Gold layer contains the finalized customer dimension
used for analytics and reporting.


In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")


# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

## Merge Acquired Customers into Unified Customer Dimension

Customer records from the acquired company are merged into the
parent company’s customer dimension using Delta Lake merge logic.


In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()